# Notebook 0: Build labelled building footprints

This shared notebook converts UNOSAT point assessments and Microsoft Building
Footprints into the per-date binary building tables used by both the
Sentinel-1 SAR and Planet pipelines. It performs five steps:

1. Read the selected UNOSAT geodatabases and pool the assessment rounds.
2. Download Microsoft Building Footprints for each study area.
3. Remove footprint artifacts below the configured area threshold.
4. Join buffered damage points to buildings once per assessment date.
5. Validate row alignment, labels, prevalence and temporal transitions.

The UNOSAT assessments are the source labels. The files written to
`shared/data/labeled_footprints` are building footprints with those labels
attached. Gaza is used for model development; the other registered cities are
whole-city holdouts for the Sentinel-1 pipeline.


## 1. Setup

The shared data is independent of either image sensor:

```
War-Damage-Detection/
    shared/
        project_config.py
        0_build_labeled_footprints.ipynb
        data/
            unosat/                  <- existing unzipped UNOSAT .gdb folders
            labeled_footprints/      <- written by this notebook
            footprints_cache/        <- downloaded Microsoft footprint tiles
```

`ONLY_GDBS` can limit a run to selected filenames. Existing cached footprint
downloads are reused.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q mercantile pyogrio


In [ ]:
import os
import sys
import re
import glob
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import mercantile
import pyogrio
from shapely import geometry
from shapely.ops import transform
from tqdm.auto import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/War-Damage-Detection"
SHARED_DIR = os.path.join(PROJECT_ROOT, "shared")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from shared.project_config import CITY_REGISTRY, POSITIVE_DAMAGE_CODES

UNOSAT_DIR = os.path.join(SHARED_DIR, "data", "unosat")
OUT_DIR = os.path.join(SHARED_DIR, "data", "labeled_footprints")
FOOTPRINT_CACHE = os.path.join(SHARED_DIR, "data", "footprints_cache")
legacy_unosat = os.path.join(PROJECT_ROOT, "Data", "UNOSAT")
legacy_footprints = os.path.join(PROJECT_ROOT, "Data", "own_footprints")
if not os.path.isdir(UNOSAT_DIR) and os.path.isdir(legacy_unosat):
    raise FileNotFoundError(
        f"Move {legacy_unosat} to {UNOSAT_DIR} before running this notebook.")
if not os.path.isdir(OUT_DIR) and os.path.isdir(legacy_footprints):
    raise FileNotFoundError(
        f"Move {legacy_footprints} to {OUT_DIR} before running this notebook.")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FOOTPRINT_CACHE, exist_ok=True)

# Microsoft's index of building-footprint tiles, with one row per quadkey.
MS_LINKS_URL = ("https://minedbuildings.z5.web.core.windows.net/"
                "global-buildings/dataset-links.csv")

# None processes all GDB files; a filename list restricts the run.
# Example: ["CE20130604SYR_Raqqa_Deir.gdb"]
ONLY_GDBS = None
INSPECT_GDBS = False

gdb_paths = sorted(glob.glob(os.path.join(UNOSAT_DIR, "*.gdb")))
if ONLY_GDBS is not None:
    keep = set(ONLY_GDBS)
    missing = keep - {os.path.basename(p) for p in gdb_paths}
    if missing:
        print(f"ONLY_GDBS names not found in {UNOSAT_DIR}: {sorted(missing)}")
    gdb_paths = [p for p in gdb_paths if os.path.basename(p) in keep]

print(f"{len(gdb_paths)} geodatabases selected"
      f"{' (ONLY_GDBS active)' if ONLY_GDBS is not None else ''} in {UNOSAT_DIR}:")
for path in gdb_paths:
    print("  ", os.path.basename(path))
if not gdb_paths:
    print(f"No geodatabases selected in {UNOSAT_DIR}. Expected unzipped .gdb "
          f"directories; ONLY_GDBS={ONLY_GDBS!r}.")


## 2. Label construction rules

The AOI, footprint filter, spatial tolerance and binary target definition
live together here. Notebook 1 includes the target definition in processed
file names, preventing labels made with different damage codes from sharing
the same patch arrays.

| setting | what it does |
|---|---|
| `GAZA_AOI` | limits footprints and assessment points to the Gaza Strip |
| `MIN_AREA_M2 = 50` | removes footprints with area at or below 50 m² |
| `SNAP_M = 10` | tolerates point and polygon-location error in the join |
| `POSITIVE_DAMAGE_CODES = (1, 2, 3)` | counts destroyed, severe and moderate damage as positive |

**Area of interest.** The Gaza Strip is a narrow band running southwest to
northeast. A rectangular extent would also include buildings in Israeli
territory that were not assessed by UNOSAT, adding them to the denominator
and reducing the observed damaged share. The project therefore uses a
20-vertex outline.

**The area filter.** Microsoft's footprints are model-derived, and the layer
contains artifacts: sheds, vehicles, wall fragments, shadows. Very small model-derived polygons are often sheds, vehicles, wall fragments
or shadows rather than reliable building footprints. The pipeline drops
everything at or below 50 m².

**The snap tolerance.** UNOSAT points do not always fall inside the associated
building. Polygon boundaries and point locations both carry error, and a strict
"point inside polygon" test can miss them. The pipeline buffers every point
by 10 m and keeps any building the circle touches. Only points whose
class is in `POSITIVE_DAMAGE_CODES` create positive labels here; all other
codes leave the building intact (`class=0`).


In [ ]:
GAZA_AOI = geometry.Polygon([
    [34.216324, 31.325772], [34.235206, 31.296147], [34.244819, 31.276782],
    [34.252373, 31.253305], [34.263702, 31.231290], [34.282928, 31.233932],
    [34.340263, 31.266805], [34.378715, 31.301720], [34.368759, 31.360958],
    [34.374939, 31.379426], [34.400001, 31.403458], [34.428894, 31.431292],
    [34.461509, 31.453846], [34.498245, 31.486057], [34.515411, 31.500694],
    [34.551822, 31.516500], [34.572078, 31.544593], [34.540492, 31.559514],
    [34.490367, 31.596075], [34.376041, 31.483422],
])

MIN_AREA_M2 = 50    # keep footprints strictly larger than this
SNAP_M = 10         # a damage point labels any building within this distance

# The shared target definition is 1=destroyed, 2=severely damaged,
# and 3=moderately damaged. All other codes remain class=0.

# -----------------------------------------------------------------------------

ADMIN_FIELDS = ("Governorate", "Municipality", "Neighborhood", "SiteID")

# Each city lists the exact (filename, layer) pairs to read. Some releases
# contain layers for several cities, while the Mosul assessments are split
# across several layers. Gaza uses sources=None with a filename match because
# each of its selected files contains one point layer with several rounds.
# City roles and war_start dates are defined in shared.project_config.
CITY_SOURCES = {
    "Gaza": {
        "sources": None,
        "match": "GazaStrip",
        "aoi": GAZA_AOI,
    },
    "Raqqa": {
        "sources": [
            ("CE20130604SYR_Raqqa_Deir.gdb", "Damage_Sites_Raqqa_CDA_Ex_wroads"),
        ],
        "aoi": None,               # bounding box of the newest assessment
    },
    "Mosul": {
        "sources": [
            ("CE20140613IRQ_Mosul_damage_assessment.gdb", "Mosul_Damage_Sites_20170611"),
            ("CE20140613IRQ_Mosul_damage_assessment.gdb", "Mosul_Damage_Sites_20170616"),
            ("CE20140613IRQ_Mosul_damage_assessment.gdb", "Mosul_Damage_Sites_20170630"),
            ("Damage_assessment_Mosul_20170804.gdb", "Mosul_Damage_Sites_20170804"),
        ],
        "aoi": None,
    },
    "Chernihiv": {
        "sources": [
            ("CE20220223UKR_UNOSAT_Chernihiv_Damage.gdb", "Chernihiv_28April2022_CDA"),
        ],
        "aoi": None,
    },
    "Rubizhne": {
        "sources": [
            ("UNOSAT_CE20220223UKR_Rubizhne_CDA_20220709.gdb", "Rubizhne_CDA_20220709"),
        ],
        "aoi": None,               # bounding box of the newest assessment
    },
}


def resolve_sources(city):
    """Return the selected (path, layer) pairs for one city.

    Explicit sources are limited to the selected filenames. For entries with
    sources=None, filenames are filtered by the optional match substring and
    the point layer is detected when the file is read.
    """
    spec = CITY_SOURCES[city]
    if spec["sources"] is None:
        pattern = spec.get("match")
        paths = (gdb_paths if pattern is None
                 else [p for p in gdb_paths if pattern in os.path.basename(p)])
        return [(p, None) for p in paths]
    available = {os.path.basename(p) for p in gdb_paths}
    return [(os.path.join(UNOSAT_DIR, fname), layer)
            for fname, layer in spec["sources"] if fname in available]


unregistered = sorted(set(CITY_SOURCES) - set(CITY_REGISTRY))
if unregistered:
    raise KeyError(f"Cities missing from shared.project_config.CITY_REGISTRY: {unregistered}")
active_cities = [c for c in CITY_SOURCES if resolve_sources(c)]
skipped_cities = [c for c in CITY_SOURCES if c not in active_cities]
print(f"AOI bounds {[round(v, 3) for v in GAZA_AOI.bounds]}, "
      f"{len(GAZA_AOI.exterior.coords) - 1} vertices")
print(f"Positive UNOSAT damage codes: {POSITIVE_DAMAGE_CODES}; "
      "all other codes remain intact")
print(f"Active cities: {active_cities}")
if skipped_cities:
    print(f"Cities without selected sources: {skipped_cities}")


### Source-specific notes

The source lists account for differences in how the UNOSAT releases are
organized:

* **Raqqa** (`CE20130604SYR_Raqqa_Deir.gdb`) also contains a
  `Damage_Sites_Deir_ez_Zor` point layer for a different city. Detecting only
  the first point layer could select the wrong data, so the Raqqa layer is
  pinned explicitly.
* **Mosul** ships as two files: `CE20140613IRQ_Mosul_damage_assessment.gdb`
  contains three single-date point layers (`..._20170611`, `_20170616` and
  `_20170630`), while `Damage_assessment_Mosul_20170804.gdb` contains a
  fourth. All four layers are named explicitly so that no assessment date is
  omitted. The files use a compound `WGS 84 + EGM96 height` CRS. During
  import, geometries are reprojected to `EPSG:4326` and their Z coordinates
  are removed.
* **Chernihiv's file** (`CE20220223UKR_UNOSAT_Chernihiv_Damage.gdb`) is a
  regional Ukraine export containing separate layers for many cities,
  including Bucha, Irpin, Kharkiv, Sumy, Donetsk and Mariupol. Naming
  `Chernihiv_28April2022_CDA` explicitly prevents points from other cities
  from entering the pool. The layer is
  post-invasion and in the standard multi-round format.
* **Rubizhne** (`UNOSAT_CE20220223UKR_Rubizhne_CDA_20220709.gdb`) contains
  one relevant point layer (`Rubizhne_CDA_20220709`) in the standard
  multi-round schema. The AOI is a padded bounding box around the newest
  assessment, consistent with the other holdout cities.


## 3. Read one geodatabase

UNOSAT stores repeat assessments **side by side** rather than as extra rows:
round 1 is `Main_Damage_Site_Class` + `SensorDate`, round 2 is
`Main_Damage_Site_Class_2` + `SensorDate_2`, and so on. A point only has a
value in round *r* once it has been assessed, so the number of assessed points
grows with every round.

`rounds_in_gdb` turns that wide layout into a list of `(date, points)` pairs,
so releases with different numbers of rounds can be pooled in the next step.


In [ ]:
def inspect_gdb(path):
    """Print every layer in a geodatabase with its geometry type and size."""
    print(f"\n{os.path.basename(path)}")
    for name, geom_type in pyogrio.list_layers(path):
        info = pyogrio.read_info(path, layer=name)
        print(f"   {name:35s} {str(geom_type):12s} {info['features']:>9,} features")


def detect_rounds(columns):
    """Find the {round number: column names} groups in a UNOSAT layer."""
    rounds = {}
    for c in columns:
        m = re.fullmatch(r"Main_Damage_Site_Class(?:_(\d+))?", c)
        if not m:
            continue
        n = int(m.group(1)) if m.group(1) else 1
        suffix = "" if n == 1 else f"_{n}"
        rounds[n] = {"class": c,
                     "date": f"SensorDate{suffix}",
                     "confidence": f"ConfidenceID{suffix}"}
    for cols in rounds.values():                    # Keep only existing columns
        for key in ["date", "confidence"]:
            if cols[key] not in columns:
                cols[key] = None
    return dict(sorted(rounds.items()))


def rounds_in_gdb(path, layer=None):
    """[(assessment date, points)] for every round in one geodatabase."""
    if layer is None:
        pl = [n for n, g in pyogrio.list_layers(path) if g and "Point" in str(g)]
        if not pl:
            print(f"   {os.path.basename(path)}: no point layer, skipped")
            return []
        layer = pl[0]

    pts = pyogrio.read_dataframe(path, layer=layer).to_crs("EPSG:4326")
    if pts.geometry.has_z.any():        # UNOSAT points carry an elevation
        pts["geometry"] = pts.geometry.apply(
            lambda g: transform(lambda x, y, z=None: (x, y), g))

    rounds = detect_rounds(pts.columns)
    if not rounds:
        print(f"   {os.path.basename(path)}: no Main_Damage_Site_Class column. "
              f"Columns are {list(pts.columns)[:12]}")
        return []

    extra = [f for f in ADMIN_FIELDS if f in pts.columns]
    out = []
    for cols in rounds.values():
        assessed = pts[pts[cols["class"]].notna()]
        if cols["date"] is None or assessed.empty:
            continue
        dates = assessed[cols["date"]].dropna()
        if dates.empty:
            continue
        date = dates.dt.date.mode().iloc[0]   # some records contain date typos

        frame = assessed[["geometry", cols["class"]] + extra].rename(
            columns={cols["class"]: "damage_class"})
        frame["confidence"] = (assessed[cols["confidence"]].to_numpy()
                               if cols["confidence"] else np.nan)
        out.append((date, frame.reset_index(drop=True)))
    return out


if INSPECT_GDBS:
    for p in gdb_paths:
        inspect_gdb(p)
else:
    print("Layer inspection: disabled")


## 4. Pool the geodatabases into one damage dataset

Each UNOSAT release is cumulative, so the same assessment date appears in
several `.gdb` folders. Records are keyed by the **assessment date**. When a
date occurs more than once, the version with more points is retained. Equal-size
candidates are resolved deterministically by release date and source name.

The result is one long table, one row per damage point per assessment date,
saved to `own_footprints/` so the geodatabases never need re-reading.


In [ ]:
def release_date(name):
    """Pull the release date out of a filename like ..._06September2024.gdb."""
    m = re.search(r"(\d{1,2})([A-Za-z]{3,9})(\d{4})", name)
    if not m:
        return None
    day, month, year = m.groups()
    for fmt, text in (("%b", month[:3]), ("%B", month)):
        try:
            return datetime.strptime(f"{day}{text}{year}", f"%d{fmt}%Y").date()
        except ValueError:
            continue
    return None


def collect_damage(city, sources):
    """Pool all assessment rounds for one city into one deduplicated dataset."""
    print(f"\n{city}: reading {len(sources)} (file, layer) source(s)")

    best, log = {}, []        # assessment date -> (points, source, rank)
    for path, layer in sources:
        name = os.path.basename(path)
        found = rounds_in_gdb(path, layer)
        rel = release_date(name)
        newest = max((d for d, _ in found), default=None)
        log.append({"file": name, "layer": layer or "(auto)", "release": rel,
                    "rounds": len(found), "newest_round": newest,
                    "matches": (rel == newest) if rel and newest else None})
        for date, frame in found:
            rank = (len(frame), rel or datetime.min.date(), name, layer or "")
            if date not in best or rank > best[date][2]:
                best[date] = (frame, name, rank)

    print(pd.DataFrame(log).to_string(index=False))
    if not best:
        raise ValueError(f"{city}: no assessment rounds found")

    parts = []
    for date, (frame, source, _) in sorted(best.items()):
        f = frame.copy()
        f["date"] = pd.Timestamp(date)
        f["source_gdb"] = source
        parts.append(f)
    damage = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True),
                              crs="EPSG:4326")

    aoi = CITY_SOURCES[city]["aoi"]
    if aoi is None:                    # Use a padded bounding box as the AOI
        newest = damage[damage["date"] == damage["date"].max()]
        minx, miny, maxx, maxy = newest.total_bounds
        aoi = geometry.box(minx - 0.005, miny - 0.005, maxx + 0.005, maxy + 0.005)

    before = len(damage)
    damage = damage[damage.within(aoi)]

    path = os.path.join(OUT_DIR, f"{city.lower()}_unosat_damage_points.parquet")
    damage.to_parquet(path)

    print(f"\n{city}: {len(best)} assessment dates, {len(damage):,} point-date "
          f"rows ({before - len(damage):,} fell outside the AOI)")
    print(f"   saved {os.path.basename(path)}")
    print(f"   damage classes: "
          f"{sorted(pd.unique(damage['damage_class'].dropna()))}")
    print(damage.groupby(damage["date"].dt.date)
                .agg(n_assessed=("damage_class", "size"),
                     source=("source_gdb", "first")).to_string())
    return damage, aoi


damage_by_city, aoi_by_city = {}, {}
for city in active_cities:
    damage_by_city[city], aoi_by_city[city] = collect_damage(city, resolve_sources(city))


## 5. Download the building footprints

Microsoft publishes footprints as tiles indexed by **quadkey**, a way of
naming map tiles at a given zoom level. The code identifies the zoom-9 tiles
that intersect the AOI, downloads them and retains the buildings inside it.

Tiles are read in **chunks** because one zoom-9 tile over a dense region can
contain millions of buildings. Buildings are retained when they **intersect**
the AOI, which preserves footprints crossing the boundary. Before identifiers
are assigned, footprints are **sorted by location** so repeated runs give the
same building the same identifier. Notebook 2 relies on these stable
identifiers when matching labels across dates.

The cache filename includes a fingerprint of the AOI. Changing the outline
therefore creates a new cache instead of silently reusing incompatible rows.


In [ ]:
def download_footprints(city, aoi, chunksize=200_000):
    """Microsoft Building Footprints inside one AOI, cached on Drive."""
    aoi_tag = hashlib.sha256(aoi.wkb).hexdigest()[:12]
    cache = os.path.join(
        FOOTPRINT_CACHE, f"{city.lower()}_z9_{aoi_tag}_footprints.parquet")
    if os.path.exists(cache):
        gdf = gpd.read_parquet(cache)
        print(f"{city}: {len(gdf):,} footprints loaded from cache")
        return gdf

    minx, miny, maxx, maxy = aoi.bounds
    quad_keys = sorted({mercantile.quadkey(t) for t in
                        mercantile.tiles(minx, miny, maxx, maxy, zooms=9)})
    print(f"{city}: AOI spans {len(quad_keys)} tiles: {quad_keys}")

    links = pd.read_csv(MS_LINKS_URL, dtype=str)
    parts = []
    for qk in tqdm(quad_keys, desc=f"{city} tiles"):
        urls = links.loc[links["QuadKey"] == qk, "Url"].tolist()
        if not urls:
            print(f"   quadkey {qk} not in the Microsoft index, skipping")
            continue
        for url in urls:
            for chunk in pd.read_json(url, lines=True, chunksize=chunksize):
                g = gpd.GeoDataFrame(
                    {"geometry": chunk["geometry"].apply(geometry.shape)},
                    crs="EPSG:4326")
                g = g[g.intersects(aoi)]
                if len(g):
                    parts.append(g)

    if not parts:
        raise RuntimeError(f"{city}: Microsoft footprint query returned no rows for the AOI")

    gdf = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs="EPSG:4326")
    gdf = gdf[~gdf.geometry.to_wkb().duplicated()].reset_index(drop=True)

    # Project centroids to metres before applying a stable spatial order.
    metric = gdf.estimate_utm_crs()
    c = gdf.to_crs(metric).centroid
    gdf = (gdf.assign(_x=c.x, _y=c.y).sort_values(["_y", "_x"])
              .drop(columns=["_x", "_y"]).reset_index(drop=True))

    gdf.to_parquet(cache)
    print(f"{city}: {len(gdf):,} footprints downloaded and cached")
    return gdf


# Download footprints only for cities with pooled damage data.
footprints_by_city = {city: download_footprints(city, aoi_by_city[city])
                      for city in damage_by_city}


## 6. Clean the footprints, then join

`prepare_footprints` applies the area filter once per city, and
`build_date_labels` runs the spatial join once per assessment date.

The join selects the configured positive UNOSAT codes, buffers those points
into 10 m circles and marks each intersecting building as damaged.

A circle can intersect more than one adjacent building, and it can match a
point that lies slightly outside its associated footprint. This preserves the
configured spatial tolerance rather than assigning every point only to its
nearest footprint.

Everything happens in a **metre-based projection** (UTM), so `SNAP_M` and
`MIN_AREA_M2` mean real metres and square metres. Only the `.geo` column is
converted back to longitude and latitude at the end, because that is what
notebook 1 reads.

Each row of the output carries:

* `class`: 1 if a point with a configured positive code touched the building
* `damage_pts`: number of selected positive points touching the building; it
  directly determines `class` and is excluded from model features
* `severity`: most severe selected positive UNOSAT class on the building
* `confidence`: minimum recorded confidence code among matched points
* `area`: footprint area in square metres
* `system:index`: stable building identifier, identical across all dates


In [ ]:
def prepare_footprints(city, footprints):
    """Project to metres, measure area, drop artifacts, assign stable ids."""
    utm = footprints.estimate_utm_crs()
    fp = footprints.to_crs(utm).copy()
    fp["area"] = fp.area.round(2)

    before = len(fp)
    fp = fp[fp["area"] > MIN_AREA_M2].reset_index(drop=True)
    fp["system:index"] = [f"{city.lower()}_{i}" for i in range(len(fp))]

    print(f"\n{city}: {before:,} footprints, {before - len(fp):,} dropped at "
          f"<= {MIN_AREA_M2} m2, {len(fp):,} kept  (CRS {utm.to_string()})")
    return fp


def build_date_labels(city, footprints, points, date):
    """Label selected UNOSAT classes as damaged and save the CSV."""
    # Only configured positive codes create damage circles. Points with all
    # other codes stay in `points` for auditing but leave buildings intact.
    positive = points[points["damage_class"].isin(POSITIVE_DAMAGE_CODES)].copy()
    circles = positive[["damage_class", "confidence"]].copy()
    circles["geometry"] = positive.buffer(SNAP_M)
    circles = gpd.GeoDataFrame(circles, geometry="geometry", crs=points.crs)

    joined = gpd.sjoin(footprints[["geometry", "system:index"]], circles,
                       how="left", predicate="intersects")

    # Aggregate multiple circles touching the same building back to one row.
    stats = joined.groupby("system:index").agg(
        damage_pts=("index_right", "count"),
        severity=("damage_class", "min"),        # 1 = most severe
        confidence=("confidence", "min"))

    out = footprints[["system:index", "area", "geometry"]].merge(
        stats, on="system:index", how="left")
    out["damage_pts"] = out["damage_pts"].fillna(0).astype(int)
    out["class"] = (out["damage_pts"] > 0).astype(int)

    # Notebook 1 reads lon/lat geometry as a GeoJSON string in ".geo".
    lonlat = gpd.GeoSeries(out["geometry"], crs=footprints.crs).to_crs("EPSG:4326")
    out[".geo"] = [json.dumps(geometry.mapping(g)) for g in lonlat]
    out = out.drop(columns=["geometry"])

    date_str = pd.Timestamp(date).strftime("%Y%m%d")
    out.to_csv(os.path.join(OUT_DIR, f"{city}_{date_str}_1_footprints.csv"),
               index=False)

    intersections = int(joined["index_right"].notna().sum())
    excluded = len(points) - len(positive)
    print(f"   {date_str}: {out['class'].mean() * 100:5.2f} percent damaged, "
          f"{len(positive):,}/{len(points):,} points use positive codes, "
          f"{excluded:,} other-code points kept intact, "
          f"{intersections:,} building-point intersections")
    return date_str


def build_city(city):
    footprints = prepare_footprints(city, footprints_by_city[city])
    damage = damage_by_city[city].to_crs(footprints.crs)
    dates = sorted(damage["date"].unique())
    print(f"{city}: {len(dates)} assessment dates")
    return [build_date_labels(city, footprints, damage[damage["date"] == d], d)
            for d in dates]


dates_by_city = {city: build_city(city) for city in footprints_by_city}

print("\nAssessment dates and CITY_REGISTRY coverage:")
for city, dates in dates_by_city.items():
    configured = list(CITY_REGISTRY[city]["label_dates"])
    unused = sorted(set(dates) - set(configured))
    missing = sorted(set(configured) - set(dates))
    print(f"  {city}: discovered={dates}; configured={configured}")
    if unused:
        print(f"    not selected downstream: {unused}")
    if missing:
        print(f"    configured but absent from this run: {missing}")


## 7. Internal label QA

Every date for a city must have the same unique footprint rows in the same
order. The checks below also validate the binary target against `damage_pts`,
summarize prevalence and report each consecutive-date transition table.

Physical destruction is cumulative, but raw UNOSAT releases can revise, omit
or reclassify earlier points. A falling prevalence or a `1-to-0` transition is
therefore recorded as a diagnostic rather than treated as a construction
failure. Cumulative propagation, when enabled, is applied later in notebook 2.


In [ ]:
REQUIRED_LABEL_COLUMNS = {
    "system:index", "area", "damage_pts", "severity",
    "confidence", "class", ".geo",
}
qa_rows, transition_rows = [], []

for city, dates in dates_by_city.items():
    tables, reference = {}, None
    for date in dates:
        path = os.path.join(OUT_DIR, f"{city}_{date}_1_footprints.csv")
        table = pd.read_csv(path, low_memory=False)
        missing_columns = REQUIRED_LABEL_COLUMNS - set(table.columns)
        if missing_columns:
            raise ValueError(f"{city} {date}: missing {sorted(missing_columns)}")
        if table["system:index"].isna().any() or table[".geo"].isna().any():
            raise ValueError(f"{city} {date}: missing id or geometry")
        if table["system:index"].duplicated().any():
            raise ValueError(f"{city} {date}: duplicate building ids")
        classes = set(table["class"].dropna().astype(int).unique())
        if not classes <= {0, 1}:
            raise ValueError(f"{city} {date}: invalid classes {sorted(classes)}")
        derived = (table["damage_pts"].to_numpy(int) > 0).astype(int)
        if not np.array_equal(derived, table["class"].to_numpy(int)):
            raise ValueError(f"{city} {date}: class disagrees with damage_pts")
        bad_severity = ((table["class"] == 1) &
                        ~table["severity"].isin(POSITIVE_DAMAGE_CODES))
        if bad_severity.any():
            raise ValueError(f"{city} {date}: positive row has invalid severity")

        if reference is None:
            reference = table[["system:index", "area", ".geo"]].copy()
        else:
            if not np.array_equal(
                    reference["system:index"], table["system:index"]):
                raise ValueError(f"{city} {date}: footprint id/order drift")
            if not np.array_equal(reference[".geo"], table[".geo"]):
                raise ValueError(f"{city} {date}: footprint geometry drift")
            if not np.allclose(reference["area"], table["area"], equal_nan=True):
                raise ValueError(f"{city} {date}: footprint area drift")

        tables[date] = table
        qa_rows.append({
            "city": city, "date": date, "buildings": len(table),
            "damaged": int(table["class"].sum()),
            "damaged_fraction": float(table["class"].mean()),
            "positive_points": int(table["damage_pts"].sum()),
        })

    for previous, current in zip(dates, dates[1:]):
        y0 = tables[previous]["class"].to_numpy(int)
        y1 = tables[current]["class"].to_numpy(int)
        counts = {(a, b): int(((y0 == a) & (y1 == b)).sum())
                  for a in (0, 1) for b in (0, 1)}
        transition_rows.append({
            "city": city, "from_date": previous, "to_date": current,
            "intact_to_intact": counts[(0, 0)],
            "intact_to_damaged": counts[(0, 1)],
            "damaged_to_intact": counts[(1, 0)],
            "damaged_to_damaged": counts[(1, 1)],
        })
        if counts[(1, 0)]:
            print(f"{city} {previous} to {current}: "
                  f"{counts[(1, 0)]:,} raw labels changed 1-to-0")

QA_SUMMARY = pd.DataFrame(qa_rows)
TRANSITIONS = pd.DataFrame(transition_rows)
display(QA_SUMMARY)
if not TRANSITIONS.empty:
    display(TRANSITIONS)

for city, frame in QA_SUMMARY.groupby("city", sort=False):
    plt.figure(figsize=(7, 3))
    plt.plot(frame["date"], 100 * frame["damaged_fraction"], marker="o")
    plt.ylabel("damaged footprints (%)")
    plt.title(f"{city}: raw UNOSAT-derived label prevalence")
    plt.xticks(rotation=45)
    plt.grid(alpha=.2)
    plt.tight_layout()
    plt.show()
